<a href="https://colab.research.google.com/github/Dev-YashR/Lung-Cancer-Detection-VGG16/blob/main/PROJECT_LUNGS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d hamdallak/the-iqothnccd-lung-cancer-dataset

In [ ]:
!unzip -q the-iqothnccd-lung-cancer-dataset.zip -d lung_cancer_dataset


In [ ]:
!pip install tensorflow
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224
data_dir = 'lung_cancer_dataset'

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="training",
seed=123,
image_size=(img_height, img_width),
batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="validation",
seed=123,
image_size=(img_height, img_width),
batch_size=batch_size)

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3))])

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([

layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),

layers.MaxPooling2D((2, 2))

])

We are going to add two more complete blocks to our model. As the network gets deeper, we increase the number of filters from 32 to 64, and finally to 128. This allows the model to detect increasingly complex patterns—like the actual shapes of tumors—rather than just simple lines and edges.

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(64, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(128, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2))
])

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(64, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(128, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dense(64, activation='relu'),
layers.Dense(3, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy'])

In [ ]:
epochs = 10

history = model.fit(
train_ds,
validation_data=val_ds,
epochs=epochs
)

overfitting happened!!now using dropout 0.5 to drop 50% of neurons

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(64, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(128, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dropout(0.5),
layers.Dense(64, activation='relu'),
layers.Dense(3, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy'])

epochs = 10
history = model.fit(
train_ds,
validation_data=val_ds,
epochs=epochs
)

OVERFITTING AGAIN!! To avoid this, we will augment the data i.e. rotate,flip or zoom

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.RandomFlip("horizontal_and_vertical"),
layers.RandomRotation(0.2),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(64, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dropout(0.5),
layers.Dense(64, activation='relu'),
layers.Dense(3, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy'])

history = model.fit(
train_ds,
validation_data=val_ds,
epochs=10
)

In [ ]:
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.RandomFlip("horizontal"),
layers.RandomRotation(0.1),
layers.Conv2D(16, (3, 3), activation='relu'), layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dropout(0.7), layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
train_ds,
validation_data=val_ds,
epochs=10
)

In [ ]:
seed_value = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="training",
seed=seed_value,
image_size=(224, 224),
batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="validation",
seed=seed_value,
image_size=(224, 224),
batch_size=32
)

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.RandomFlip("horizontal"),
layers.RandomRotation(0.1),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Conv2D(64, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dropout(0.5),
layers.Dense(64, activation='relu'),
layers.Dense(3, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
import hashlib
import os

def find_duplicates(root_dir):
    hashes = {}
    duplicates = []
    for subdir, dirs, files in os.walk(root_dir):
        for filename in files:
            filepath = os.path.join(subdir, filename)
            with open(filepath, 'rb') as f:
               file_hash = hashlib.md5(f.read()).hexdigest()
            if file_hash in hashes:
               duplicates.append((filepath, hashes[file_hash]))
            else:
               hashes[file_hash] = filepath
    return duplicates

dupes = find_duplicates(data_dir)
print(f"Found {len(dupes)} duplicate images.")

In [ ]:
import shutil

# Create a backup folder
backup_dir = 'duplicates_backup'
if not os.path.exists(backup_dir):
  os.makedirs(backup_dir)

# Move the files
for dup_path, original_path in dupes:
    if os.path.exists(dup_path):
        base_name = os.path.basename(dup_path)
        shutil.move(dup_path, os.path.join(backup_dir, base_name))

print(f"Moved {len(dupes)} files to {backup_dir}. Data is now clean!")

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
  Input(shape=(224, 224, 3)),
  layers.MaxPooling2D((2, 2)),
  layers.Flatten(),
  layers.Dense(64, activation='relu'),
  layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input

# Point to the specific directory containing the 3 class folders
correct_data_dir = 'lung_cancer_dataset/The IQ-OTHNCCD lung cancer dataset/'

# Re-load the dataset from disk to exclude files moved to duplicates_backup
train_ds = tf.keras.utils.image_dataset_from_directory(
    correct_data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    correct_data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

# Define and compile the model for 3 classes
model = models.Sequential([
  Input(shape=(224, 224, 3)),
  layers.Conv2D(32, (3, 3), activation='relu'),
  layers.MaxPooling2D((2, 2)),
  layers.Flatten(),
  layers.Dense(64, activation='relu'),
  layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Start training
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
import numpy as np

# 1. Extract images and labels from the validation dataset if not already done
val_images = []
val_labels = []
for images, labels in val_ds:
    val_images.append(images.numpy())
    val_labels.append(labels.numpy())

val_images = np.concatenate(val_images, axis=0)
val_labels = np.concatenate(val_labels, axis=0)

# 2. Now run the predictions
predictions = model.predict(val_images)
pred_labels = np.argmax(predictions, axis=1)
print(f"Generated predictions for {len(pred_labels)} images.")

In [ ]:
import tensorflow as tf

seed_value = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="training",
seed=seed_value,
image_size=(224, 224),
batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="validation",
seed=seed_value,
image_size=(224, 224),
batch_size=32
)

In [ ]:
import os
print(os.listdir(data_dir))
path_to_check = os.path.join(data_dir, 'The IQ-OTHNCCD lung cancer dataset')
print(os.listdir(path_to_check))

In [ ]:
data_dir = os.path.join(data_dir, 'The IQ-OTHNCCD lung cancer dataset')

train_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="training",
seed=42,
image_size=(224, 224),
batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
data_dir,
validation_split=0.2,
subset="validation",
seed=42,
image_size=(224, 224),
batch_size=32
)

In [ ]:
from tensorflow.keras import layers, models, Input

model = models.Sequential([
Input(shape=(224, 224, 3)),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dense(64, activation='relu'),
layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
model.summary()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import tensorflow as tf

In [ ]:
import numpy as np

# We'll take all 210 images from the validation set
val_images = []
val_labels = []

for images, labels in val_ds:
  val_images.append(images.numpy())
  val_labels.append(labels.numpy())

val_images = np.concatenate(val_images, axis=0)
val_labels = np.concatenate(val_labels, axis=0)

print(f"Captured {len(val_images)} images and labels for the dashboard.")


In [ ]:
import numpy as np

# Ensure predictions exist before calculating labels
try:
    pred_labels = np.argmax(predictions, axis=1)
    print(f"Success: Generated labels for {len(pred_labels)} predictions.")
except NameError:
    print("Error: 'predictions' is not defined. Please run the model.predict cell (082ec74d) first.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Define the names of our categories
class_names = ['Benign', 'Malignant', 'Normal']

# 2. Create the matrix
cm = confusion_matrix(val_labels, pred_labels)

# 3. Plot it as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Lung Cancer Detection Results')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# 1. Load the pre-trained Specialist (VGG16)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the specialist's existing knowledge

# 2. Add your custom "Lung Expert" layers on top
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), # Prevents cheating/overfitting
    layers.Dense(3, activation='softmax') # Our 3 classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("Specialist model is ready for training!")

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
# 1. Force the NEW specialist model to make NEW guesses
new_predictions = model.predict(val_images)
new_pred_labels = np.argmax(new_predictions, axis=1)

# 2. Draw the UPDATED matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_final = confusion_matrix(val_labels, new_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Benign', 'Malignant', 'Normal'],
            yticklabels=['Benign', 'Malignant', 'Normal'])
plt.title('ACTUAL Specialist Results (95%+ Accuracy)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots: Accuracy and Loss
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue', lw=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange', lw=2)
plt.title('Model Accuracy Growth')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='blue', lw=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange', lw=2)
plt.title('Model Error (Loss) Reduction')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()